# Video Game Sales: EDA for an indie studio
Worksheet *video games sales worksheet.pdf* (Data Science course), worked out in Python. The calculations live in [toolkit.py](toolkit.py); this notebook follows the worksheet step by step and writes the answers under each step.

| Group | Coach | Date |
|---|---|---|
| | | |

**Objective:** analyze the video game sales dataset, perform exploratory data analysis (EDA) and, based on it, make a recommendation for an indie game studio.

**Dataset:** [data/vgsales.csv](data/vgsales.csv), one row per game per platform. Sales are in **millions of copies**. The worksheet calls the year column `Year_of_Release`; in this file it is `Year`. The original file is never changed: cleaned data is saved as `data/vgsales_copy.csv`.

In [ ]:
%load_ext autoreload
%autoreload 2
import toolkit as tk

## Step 1: Load & inspect data
Goal: understand the dataset structure.
### Open the dataset as a dataframe

In [ ]:
raw = tk.load_csv("data", "vgsales.csv")
tk.shape(raw)
raw.head()

### Column names, data types and missing values

In [ ]:
tk.column_info(raw)

In [ ]:
tk.data_issues(raw)

Two more checks the automatic scan cannot do. Games that appear more than once with the same name, platform and publisher, and the most recent years:

In [ ]:
repeated = raw[raw.duplicated(["Name", "Platform", "Publisher"], keep=False)].sort_values(["Name", "Platform", "Year"])
repeated[["Rank", "Name", "Platform", "Year", "Publisher", "Global_Sales"]]

In [ ]:
tk.aggregate(raw, "Year", "Global_Sales", agg="count", plot=None).tail(6)

### Cleaning and preparation
- **Versions of the same game**: rows with the same `Name`, `Platform` and `Publisher` are different versions or releases of one game, so they are combined into one row. Their sales are added up, `Year` becomes the first release year, and `Rank` the best rank.
- **Missing `Year`**: these rows cannot be placed in time, so they are dropped. Combining comes first, so a version without a year still adds its sales to a version that has one.
- **Years after 2016**: the data was collected in 2016; 2017 and 2020 hold only a handful of games.
- **`Year` as a whole number**: it was read as a decimal (`2006.0`) only because of the missing values. Once those rows are gone, it is stored as an integer (`2006`).
- **`Rank`** is an identifier, not a measurement.
- **Missing `Publisher`**: kept. Only the publisher analysis in Step 10 uses this column, and dropping these rows would also remove their sales.

In [ ]:
SALES = ["Global_Sales", "NA_Sales", "EU_Sales", "JP_Sales", "Other_Sales"]

combined = tk.combine(raw, ["Name", "Platform", "Publisher"], sum=SALES, lowest=["Year", "Rank"])
cleaned = tk.clean(combined, duplicates=True, required=["Year"], keep={"Year": (None, 2016)}, integer=["Year"])
data = tk.prepare(cleaned, target=None, drop=["Rank"], features=SALES)

**Notes (data issues found):**
1. Four games appear twice with the same name, platform and publisher: *Madden NFL 13* (PS3, split over two rows), *Need for Speed: Most Wanted* (X360 and PC, the 2005 and 2012 releases) and *Wii de Asobu: Metroid Prime* (Wii). Each is combined into one row with the sales added up: 8 rows become 4.
2. `Year` is missing for 271 rows and is stored as decimals (`2006.0`) because of those gaps. After combining, 270 games have no year. Dropped, and `Year` is then stored as an integer.
3. 2017 and 2020 hold only 4 games, because data collection ended in 2016. Dropped: 4 rows.
4. `Publisher` is missing for 58 games and another 203 say "Unknown". Kept, see above.
5. `Rank` is just the row's position by sales, so it is left out of the analysis.

Result: **16,320 games** (from 16,598 rows).

## Step 2: Descriptive statistics & outliers
### Step 2.1 Summary statistics
Goal: understand central tendency.

In [ ]:
tk.describe(data, ["Global_Sales"]).T

**Answer** (millions of copies):

| Mean | Median | Standard deviation | Min | Max | Q1 | Q3 |
|---|---|---|---|---|---|---|
| 0.54 | 0.17 | 1.57 | 0.01 | 82.74 | 0.06 | 0.48 |

The mean is three times the median and the standard deviation is three times the mean: a few huge hits pull the average up, while the typical game sells about 170,000 copies.

### Step 2.2 Distribution analysis (raw vs log)
`Log_Sales = LOG(Global_Sales + 1)`. Excel's `LOG` is base 10, so the `"log10"` method is used; the new column is called `Global_Sales_log10`.

In [ ]:
data = tk.add_transformed(data, "log10", ["Global_Sales"])
tk.plot(data, ["Global_Sales", "Global_Sales_log10"], kind="hist", bins=60, color_by=None,
        title="Global_Sales: raw vs log")

In [ ]:
tk.describe(data, ["Global_Sales", "Global_Sales_log10"])[["mean", "median", "std", "min", "max"]]

Skewness: 0 = symmetric, larger = a longer tail to the right.

In [ ]:
data[["Global_Sales", "Global_Sales_log10"]].skew()

**1. How does the distribution change after log transformation?**
The raw histogram is one tall bar near zero with a long, nearly invisible tail up to 82.7. After the log the tail is pulled in (the maximum drops from 82.7 to 1.9) and the bulk becomes a visible, smoothly falling curve. Skewness drops from about 17 to about 2.7: still right-skewed, but far less.

It stays piled up near 0 because most games sell well under 1 M copies, and for small values `LOG(x + 1)` is close to a plain rescaling (≈ 0.43·x). The `+ 1` mostly squeezes the big hits, not the small games.

**2. Are extreme values still dominant after transformation?**
Much less. Raw, the best seller (*Wii Sports*) is 487× the median game. After the log it is 28×. The hits still sit at the far right, but they no longer squash the rest of the chart.

**3. Which representation is more balanced and why?**
The log version. It compresses large values far more than small ones, so the typical game and the hits fit on one scale. That makes it the better basis for comparing groups and spotting patterns.

### Top 10 games by global sales

In [ ]:
tk.top(data, "Global_Sales", 10, ["Name", "Platform", "Year", "Genre", "Publisher"], compare="log10")

**Top 10:** 1. Wii Sports · 2. Super Mario Bros. · 3. Mario Kart Wii · 4. Wii Sports Resort · 5. Pokemon Red/Blue · 6. Tetris · 7. New Super Mario Bros. · 8. Wii Play · 9. New Super Mario Bros. Wii · 10. Duck Hunt (all published by Nintendo).

**Do these games still dominate after log transformation?**
They keep the same ranks, because the log never changes the order. Their weight shrinks a lot, though. Raw, the top 10 games (0.06% of all games) take 4.2% of all sales, and #1 is 487× the median. On the log scale they take 0.7% and #1 is 28× the median. They are still the best sellers, but no longer in a different league.

**Decision: which should be used?** ☑ **Both**

*Justification:* use raw sales for totals, market size and shares (millions of copies are what a studio earns). Use the log values, or medians, to compare typical games between groups and to make readable charts. Otherwise a handful of Nintendo hits decide every comparison.

### Step 2.3 Outlier detection (IQR method)

In [ ]:
outlier_table, outlier_rows = tk.outliers(data, ["Global_Sales", "Global_Sales_log10"])
outlier_table

**Answer** (raw `Global_Sales`): IQR = 0.48 − 0.06 = **0.42** · Lower bound = 0.06 − 1.5 × 0.42 = **−0.57** · Upper bound = 0.48 + 1.5 × 0.42 = **1.11** · Number of outliers = **1,826** (11.2% of games).

Every outlier is on the high side: no game can sell less than 0. Even after the log, 1,319 games (8.1%) remain outliers.

### Step 2.4 Outlier interpretation
**Are outliers meaningful or errors?**
Meaningful. They are real hit games (*Wii Sports*, *GTA V*, *Call of Duty* ...), not typing errors. Video game sales are a hit-driven market, and these games are exactly where most of the money is. One in nine games is an "outlier", which shows the distribution is skewed, not that the data is wrong.

### Step 2.5 Final decision
**How should outliers be handled?** ☑ **Keep**

*Justification:* removing them would delete the most important part of the market and understate every total (platform, genre and year sales). To describe the typical game, use the median or the log values instead of removing rows.

In [ ]:
data = tk.handle_outliers(data, ["Global_Sales"], action="keep")

## Step 3: Time trend analysis
Pivot of `Year` against `Global_Sales` (total, plus number of games and average per game), and a line chart.

In [ ]:
per_year = tk.aggregate(data, "Year", "Global_Sales", agg="sum")
per_year

In [ ]:
tk.aggregate(data, "Year", "Global_Sales", agg="count", plot="line");

**Peak years:** **2008** (679 M copies), **2009** (667 M) and **2007** (611 M). Releases peak in the same years: 1,428 games in 2008 and 1,431 in 2009.

Sales grow from the 1980s, jump in the mid-1990s and peak in the Wii / DS / PS3 / Xbox 360 generation. From 2010 they decline. Part of that decline is real. Part comes from the dataset: it counts physical retail sales, so the move to digital downloads is not visible. The low 2016 total is also incomplete, because data collection ended that year.

## Step 4: Platform analysis

In [ ]:
tk.aggregate(data, "Platform", "Global_Sales", agg="sum", top=10)

**Top platforms (total sales):** 1. **PS2** (1,233 M) · 2. **Xbox 360** (970 M) · 3. **PS3** (949 M). The Wii (910 M) follows closely.

Totals reward platforms with long lives and big libraries. Per game, recent consoles do better: PS4 (0.83 M per game) and X360 (0.79 M) beat the PS2 (0.58 M).

## Step 5: Genre analysis

In [ ]:
tk.aggregate(data, "Genre", "Global_Sales", agg="sum")

In [ ]:
tk.aggregate(data, "Genre", "Global_Sales", agg="mean")

**Top genres (total sales):** 1. **Action** (1,723 M, 19.5%) · 2. **Sports** (1,309 M) · 3. **Shooter** (1,026 M).

The ranking changes per game. Action is #1 in total only because it has the most games (3,252). Its average game sells 0.53 M, which puts it 7th. Platform games (0.95 M) and Shooters (0.80 M) sell the most per game.

## Step 6: Platform × Genre analysis
Total sales per genre on the 8 biggest platforms, then each platform as a percentage of its own sales.

In [ ]:
tk.pivot(data, "Platform", "Genre", "Global_Sales", agg="sum", top_rows=8, plot="stacked")

In [ ]:
tk.pivot(data, "Platform", "Genre", "Global_Sales", agg="sum", top_rows=8, share="rows", plot="heatmap")

**Key observation:** the best genre depends on the platform. Each platform has its own audience:
- **Xbox 360**: Shooter is the #1 genre (29% of its sales) and Action 25%.
- **PS3**: Action 32%, Shooter 21%.
- **Wii**: Sports (32%) and Misc (24%), the family and party games. Shooters are only 3%.
- **DS**: Misc, Simulation, Role-Playing and Puzzle each take 10–17%. Puzzle is almost absent on the home consoles (≤ 1.7%).

"Action is the top genre" holds for the market as a whole, but not on the Wii or the DS. Genre and platform have to be chosen together.

## Step 7: Simpson's paradox
Compare the average sales per game of every pair of genres, overall and within each platform. A paradox is a pair where one genre wins overall but loses on most platforms.

In [ ]:
tk.simpson(data, "Genre", "Platform", "Global_Sales", agg="mean", min_count=10)

Why does this happen? Where the games of the two genres were released (share of each genre's games per platform):

In [ ]:
mix = tk.pivot(data, "Platform", "Genre", "Global_Sales", agg="count", share="columns", plot=None)
mix[["Racing", "Action"]].sort_values("Action", ascending=False).head(12)

**What contradiction did you observe?**
Overall, a **Racing** game sells more on average than an **Action** game (0.59 M vs 0.53 M per game). Platform by platform it is the other way around: **Action is higher on 11 of the 16 platforms** with enough games of both. That includes PS, PS2, PS3, PS4, Xbox 360 and PC.

The reason is where the games were released. Racing games are concentrated on the strong-selling PS and PS2 (29% of all Racing games, against 15% of Action games). Racing also has a few giant Nintendo hits (*Mario Kart* on Wii, DS and 3DS). Action games are spread much more over weak-selling platforms such as PSP, PS Vita and 3DS (16.5% of Action games vs 7% of Racing). The overall average mixes "which genre" with "which platform", and that mix flips the conclusion.

## Step 8: Visualization
The same platform × genre data as a grouped bar chart, to compare with the stacked chart of Step 6.

In [ ]:
tk.pivot(data, "Platform", "Genre", "Global_Sales", agg="sum", top_rows=5, top_columns=6, plot="grouped")

**Which chart was most useful?** ☑ **Stacked bar chart** (with ☑ *Other*: the percentage heatmap from Step 6)

*Why?* The stacked chart answers two questions at once: how big each platform is (bar height) and what it is made of (segments). With 12 genres, grouped bars become a forest of thin bars that is hard to read. They are only better for comparing one genre across platforms (e.g. Shooter on X360 vs PS3). The percentage heatmap removes platform size and shows each platform's mix directly. It is the clearest view of the Step 6 observation.

## Step 9 / 10: The market an indie studio can enter
The worksheet's last two steps are about an indie studio. An indie studio does not compete with *Wii Sports*: it needs a genre that is selling **now**, where it is not up against the biggest publishers. Two extra views, both on the recent years (2012–2016):
1. **Concentration**: how much of each genre's sales goes to its 5 biggest publishers.
2. **Sales outside the 20 biggest publishers**: how a game from a smaller publisher does per genre.

In [ ]:
recent = data[data["Year"] >= 2012]
print(f"Games released 2012-2016: {len(recent)}")
tk.aggregate(recent, "Platform", "Global_Sales", agg="sum")

In [ ]:
tk.concentration(recent, "Genre", "Publisher", "Global_Sales", top=5)

In [ ]:
biggest = tk.aggregate(recent, "Publisher", "Global_Sales", agg="sum", top=20, plot=None).index
smaller = recent[~recent["Publisher"].isin(biggest)]
print(f"Games from outside the 20 biggest publishers: {len(smaller)} of {len(recent)}")
tk.aggregate(smaller, "Genre", "Global_Sales", agg="median")

In [ ]:
tk.pivot(recent, "Genre", "Platform", "Global_Sales", agg="median", plot="heatmap")

## Step 9: Business insights
1. **Platform insights:** the all-time leaders (PS2, X360, PS3, Wii) are old generations. For 2012–2016 the market is PS3, PS4 and X360, followed by 3DS and Xbox One. PC sells little at retail (median 0.08 M), but that is partly because its sales are digital and missing from this data.
2. **Genre insights:** Action, Sports and Shooter make up 46% of all sales. Per game, Platform and Shooter games sell best, and Adventure, Strategy and Puzzle the worst. Big totals come from many releases, not from each game selling well.
3. **Simpson's paradox insight:** an overall average can point the wrong way. Racing "beats" Action overall, yet loses on most platforms, because the two genres were released on different platforms. Compare genres within one platform (or control for platform) before deciding.
4. **Marketing recommendation:** market a game per platform audience, not to "gamers" in general. Shooters for Xbox and PlayStation players, family, party and simulation games for Nintendo players. Release on the platform whose audience already buys the genre.
5. **Risk of aggregation:** totals and averages over the whole dataset are distorted by (a) a few huge hits (mostly Nintendo), (b) the number of releases per genre and (c) the platform mix (Simpson). This dataset also counts retail sales only, so digital indie sales are largely missing. Conclusions about "the market" can hide what holds in the segment you actually enter.

## Step 10: Indie studio strategy
**Primary niche genre: Role-Playing**

**Secondary genre: Simulation**

**Justification:**
- **Avoid the blockbuster genres.** In 2012–2016 the 5 biggest publishers own 85% of Shooter and 94% of Sports sales (Activision, EA, Take-Two). Their high averages come from AAA franchises, not from what a small studio can expect.
- **Role-Playing is the best of the big markets that smaller publishers can reach.** Outside the 20 biggest publishers, the three largest genres are Action (233 games, 28% of these sales), Adventure (214 games, 16%) and Role-Playing (106 games, 15%). Of those three, the median RPG sells **0.06 M** copies, twice as much as Action or Adventure (0.03 M), against half as many competitors. RPGs are also the most open of the higher-selling genres (top-5 publisher share 72%) and sell well on handhelds (3DS median 0.21 M).
- **Simulation is a smaller but less contested niche.** Only 35 recent games from smaller publishers, with a solid median (0.07 M) and mean (0.17 M per game). On PC, the platform that is easiest for an indie to publish on, it is the third-best genre (median 0.13 M, after Shooter and RPG). Mid-size publishers such as Focus Home Interactive are among the genre's top 3.
- **Not Action or Adventure, despite low concentration.** These are the most open markets (top-5 share 44–55%), but also the most crowded: 972 Action and 280 Adventure games in 2012–2016, with a smaller-publisher median of only 0.03 M.
- **Not Racing, despite a good median.** Smaller publishers' racing games have a median of 0.09 M, but the top 5 publishers own 81% of the genre (EA, Microsoft, Codemasters) and racing games are expensive to make (3D tracks, car licences).
- **Platform choice:** start on PC, then PS4 / 3DS, where RPG and Simulation buyers already are (Step 6). Judge success on the median game, not on the hits (Steps 2 and 7).

*Caveat:* this dataset only covers retail sales charts. Validate with digital store data (e.g. Steam) before committing.